# EuroSAT 4-Tier Land-Use Classification — End-to-End Reproduction

**CMPE 257 — Machine Learning, San José State University**
**Team:** Ekant Kapgate, Saransh Soni, Vineet Malewar, Yashashav Devalapalli Kamalraj
**Repo:** https://github.com/yashashav-dk/cmpe257-eurosat

This notebook reproduces every result reported in [`report/final_report.pdf`](../report/final_report.pdf):

- **Tier 1** Classical pixel ML (PCA + LR/SVM)
- **Tier 2** Handcrafted features (HOG + ColorHist) + classical ML
- **Tier 3** Shallow CNN (391K params) with optimizer + regularization ablations
- **Tier 4** ResNet-18 transfer learning

Run modes:
- `RUN_MODE = "quick"` (default): reduced ablations, 1 seed each, ~30 min on A100. Sanity-check reproduction.
- `RUN_MODE = "full"`: paper-equivalent runs (54 + 27 + 3 seeds each), ~5 hours on A100.

Hardware: NVIDIA GPU recommended (A100 ideal). Falls back to sklearn CPU if cuML unavailable; ShallowCNN training time scales linearly with VRAM bandwidth.


## 0. Setup

Install dependencies, configure paths, verify GPU.


In [ ]:
# Install dependencies (Colab pre-installs most; this is idempotent)
import subprocess, sys

PKGS = [
    "torch", "torchvision", "scikit-learn", "opencv-python",
    "matplotlib", "seaborn", "pandas", "numpy", "tqdm", "Pillow", "pypdf",
    "kornia",
]
for p in PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

# Optional: GPU SVM via cuML (RAPIDS). Skip if it fails — sklearn fallback.
try:
    import cuml
except ImportError:
    print("cuML not present — attempting install (skips on failure)")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cuml-cu12",
         "--extra-index-url=https://pypi.nvidia.com"],
        check=False, timeout=180,
    )
print("Dependencies installed.")


In [ ]:
# Imports + GPU check
import os, sys, time, json, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torchvision import datasets, transforms
import torchvision.models as tvmodels
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)
import cv2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected. ShallowCNN training will be slow; ResNet-18 may take hours.")

# Try cuML import
try:
    from cuml.svm import SVC as CumlSVC
    import cupy as cp
    HAS_CUML = True
    print(f"cuML available: GPU SVC enabled")
except ImportError:
    HAS_CUML = False
    print("cuML not available: classical SVM will use sklearn (slower)")

try:
    import kornia.augmentation as K
    HAS_KORNIA = True
except ImportError:
    HAS_KORNIA = False
    print("Kornia not available: data augmentation will use manual ops")


In [ ]:
# Configuration
RUN_MODE = "quick"   # "quick" or "full"

SEED = 42
SEEDS = [42, 123, 456] if RUN_MODE == "full" else [42]
NUM_CLASSES = 10
IMG_SIZE = 64
RESNET_IMG_SIZE = 224
NUM_CHANNELS = 3
TOTAL_IMAGES = 27000

CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake",
]

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
BATCH_SIZE = 64
NUM_EPOCHS_CNN = 100 if RUN_MODE == "full" else 30
NUM_EPOCHS_RESNET = 50 if RUN_MODE == "full" else 15
EARLY_STOP_PATIENCE = 10

# Working directory: try Drive, fall back to /content (Colab) or local
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive', force_remount=False)
    if os.path.ismount('/content/drive'):
        WORK_DIR = '/content/drive/MyDrive/cmpe257_eurosat_repro'
    else:
        WORK_DIR = '/content/cmpe257_eurosat_repro'
except (ImportError, ValueError):
    WORK_DIR = './cmpe257_eurosat_repro'

os.makedirs(WORK_DIR, exist_ok=True)
DATA_DIR = os.path.join(WORK_DIR, 'data', 'EuroSAT_RGB')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')
for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"RUN_MODE: {RUN_MODE}")
print(f"WORK_DIR: {WORK_DIR}")
print(f"Seeds: {SEEDS}, ShallowCNN epochs: {NUM_EPOCHS_CNN}, ResNet epochs: {NUM_EPOCHS_RESNET}")

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

seed_everything()
sns.set_theme(style="whitegrid", font_scale=1.0)


## 1. Data: Download, Split, EDA

EuroSAT RGB from Zenodo. Stratified seed-42 70/15/15 split. Per-channel statistics computed on the training split only.


In [ ]:
# Download EuroSAT RGB (94 MB, ~30s)
import urllib.request, zipfile, shutil

URL = "https://zenodo.org/record/7711810/files/EuroSAT_RGB.zip"
ZIP_PATH = os.path.join(WORK_DIR, 'data', 'EuroSAT_RGB.zip')

if not os.path.isdir(DATA_DIR) or len([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]) != 10:
    print(f"Downloading {URL} ...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print(f"Extracting ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(os.path.join(WORK_DIR, 'data'))
    if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
    nested = os.path.join(DATA_DIR, '2750')
    if os.path.isdir(nested):
        for entry in os.listdir(nested):
            shutil.move(os.path.join(nested, entry), os.path.join(DATA_DIR, entry))
        os.rmdir(nested)

classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes}
print(f"Classes: {len(classes)}, Total: {sum(counts.values())}")
for c, n in counts.items(): print(f"  {c:25s} {n}")
assert len(classes) == 10 and sum(counts.values()) >= 26000


In [ ]:
# Stratified split (deterministic)
def stratified_split_indices(dataset, seed=SEED):
    targets = list(dataset.targets)
    indices = list(range(len(dataset)))
    train_val_idx, test_idx = train_test_split(
        indices, test_size=TEST_RATIO, stratify=targets, random_state=seed)
    rel_val = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train_val_targets = [targets[i] for i in train_val_idx]
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=rel_val, stratify=train_val_targets, random_state=seed)
    return train_idx, val_idx, test_idx

raw_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
ds = datasets.ImageFolder(DATA_DIR, transform=raw_tf)
train_idx, val_idx, test_idx = stratified_split_indices(ds, seed=SEED)
print(f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")


In [ ]:
# Per-channel mean/std on TRAIN ONLY (no leakage)
print("Computing per-channel statistics on training set ...")
ch_sum = np.zeros(3); ch_sq = np.zeros(3); n_pix = 0
for i in train_idx:
    img, _ = ds[i]
    arr = img.numpy()
    ch_sum += arr.sum(axis=(1, 2))
    ch_sq += (arr ** 2).sum(axis=(1, 2))
    n_pix += arr.shape[1] * arr.shape[2]
EUROSAT_MEAN = (ch_sum / n_pix).tolist()
EUROSAT_STD = np.sqrt(ch_sq / n_pix - (ch_sum / n_pix) ** 2).tolist()
print(f"EUROSAT_MEAN = {[round(x,4) for x in EUROSAT_MEAN]}")
print(f"EUROSAT_STD  = {[round(x,4) for x in EUROSAT_STD]}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


In [ ]:
# EDA plots: class distribution + sample grid
class_counts = Counter(ds.targets)

fig, ax = plt.subplots(figsize=(12, 4))
counts = [class_counts[i] for i in range(NUM_CLASSES)]
ax.bar(CLASS_NAMES, counts, color=sns.color_palette("husl", 10))
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel("Images"); ax.set_title("EuroSAT — Class Distribution")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_class_distribution.png'), dpi=120)
plt.show()


In [ ]:
# Build numpy splits (flattened pixels) and image splits (for HOG) — cached
def get_numpy_splits(seed=SEED):
    fds = datasets.ImageFolder(DATA_DIR, transform=raw_tf)
    tr_idx, va_idx, te_idx = stratified_split_indices(fds, seed)
    def extract(idxs):
        imgs, lbls = [], []
        for i in idxs:
            img, lbl = fds[i]
            imgs.append(img.permute(1,2,0).numpy())
            lbls.append(lbl)
        return np.array(imgs), np.array(lbls)
    X_tr_img, y_tr = extract(tr_idx)
    X_val_img, y_val = extract(va_idx)
    X_te_img, y_te = extract(te_idx)
    return (X_tr_img.reshape(len(X_tr_img), -1), y_tr,
            X_val_img.reshape(len(X_val_img), -1), y_val,
            X_te_img.reshape(len(X_te_img), -1), y_te,
            X_tr_img, X_val_img, X_te_img)

print("Building numpy splits (~70 sec) ...")
t0 = time.time()
(X_tr, y_tr, X_val, y_val, X_te, y_te,
 X_tr_img, X_val_img, X_te_img) = get_numpy_splits(SEED)
print(f"Done in {time.time()-t0:.1f}s")
print(f"X_tr: {X_tr.shape}, X_val: {X_val.shape}, X_te: {X_te.shape}")

# Merge train+val for classical CV; test untouched
X_tv = np.concatenate([X_tr, X_val])
y_tv = np.concatenate([y_tr, y_val])

# GPU-resident tensors for CNN tiers (the 100x speedup)
def to_tensor(x_img, mean, std):
    t = torch.from_numpy(x_img.astype(np.float32)).permute(0,3,1,2)
    m = torch.tensor(mean).view(1,3,1,1); s = torch.tensor(std).view(1,3,1,1)
    return (t - m) / s

X_tr_t = to_tensor(X_tr_img, EUROSAT_MEAN, EUROSAT_STD).to(DEVICE)
X_val_t = to_tensor(X_val_img, EUROSAT_MEAN, EUROSAT_STD).to(DEVICE)
X_te_t = to_tensor(X_te_img, EUROSAT_MEAN, EUROSAT_STD).to(DEVICE)
y_tr_t = torch.from_numpy(y_tr).long().to(DEVICE)
y_val_t = torch.from_numpy(y_val).long().to(DEVICE)
y_te_t = torch.from_numpy(y_te).long().to(DEVICE)
print(f"GPU tensors: {X_tr_t.shape}, {X_val_t.shape}, {X_te_t.shape}")


## 2. Helpers: Unified Metric Suite + GPU Iterator

Every model passes through `compute_metrics`. The GPU iterator avoids DataLoader I/O overhead.


In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None):
    m = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class_f1": f1_score(y_true, y_pred, average=None, zero_division=0).tolist(),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if y_prob is not None:
        try: m["roc_auc"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
        except ValueError: m["roc_auc"] = None
    else:
        m["roc_auc"] = None
    return m

def gpu_iter(X, y, bs, shuffle, seed):
    n = X.shape[0]
    if shuffle:
        g = torch.Generator(device=X.device).manual_seed(seed)
        idx = torch.randperm(n, device=X.device, generator=g)
    else:
        idx = torch.arange(n, device=X.device)
    for i in range(0, n, bs):
        b = idx[i:i+bs]; yield X[b], y[b]

print("Helpers defined.")


## 3. Tier 1 — Pixel + PCA Classical

Flatten 64·64·3=12288 → StandardScaler → PCA → LR / LinearSVC / SVM-RBF.
Best result: SVM-RBF on unscaled PCA(500) ≈ 0.687 test acc.


In [ ]:
# Fit StandardScaler + PCA(500) once on training, project all splits
print("StandardScaler + PCA(500) on training set ...")
scaler_t1 = StandardScaler()
X_tr_s = scaler_t1.fit_transform(X_tr)
X_val_s = scaler_t1.transform(X_val)
X_te_s = scaler_t1.transform(X_te)
pca500 = PCA(n_components=500, random_state=SEED)
X_tr_p = pca500.fit_transform(X_tr_s)
X_val_p = pca500.transform(X_val_s)
X_te_p = pca500.transform(X_te_s)
X_tv_p = np.concatenate([X_tr_p, X_val_p])
print(f"PCA explained variance @ k=500: {np.cumsum(pca500.explained_variance_ratio_)[-1]:.4f}")


In [ ]:
# Tier 1 LR (broadened grid, post-PCA-scaled — fixes lbfgs convergence)
post_scaler = StandardScaler()
X_tv_ps = post_scaler.fit_transform(X_tv_p)
X_te_ps = post_scaler.transform(X_te_p)

t0 = time.time()
grid_lr = GridSearchCV(
    LogisticRegression(solver="lbfgs", max_iter=5000, random_state=SEED),
    {"C": [1e-3, 1e-2, 0.1, 1, 10]},
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="accuracy", n_jobs=-1,
)
grid_lr.fit(X_tv_ps[:, :200], y_tv)
y_pred = grid_lr.predict(X_te_ps[:, :200])
y_prob = grid_lr.predict_proba(X_te_ps[:, :200])
m_lr = compute_metrics(y_te, y_pred, y_prob)
print(f"LR + PCA(200) scaled: acc={m_lr['accuracy']:.4f}, F1={m_lr['macro_f1']:.4f} ({time.time()-t0:.1f}s)")


In [ ]:
# Tier 1 SVM-RBF on UNSCALED PCA(500) — best Tier 1 result
print("Training SVM-RBF on unscaled PCA(500) ...")
X_tv_un = X_tv_p.astype(np.float32)
X_te_un = X_te_p.astype(np.float32)
t0 = time.time()
if HAS_CUML:
    clf = CumlSVC(C=10.0, gamma="scale", kernel="rbf", probability=True)
    clf.fit(X_tv_un, y_tv)
    y_pred = clf.predict(X_te_un); y_pred = cp.asnumpy(y_pred) if hasattr(y_pred, "get") else np.asarray(y_pred)
    y_prob = clf.predict_proba(X_te_un); y_prob = cp.asnumpy(y_prob) if hasattr(y_prob, "get") else np.asarray(y_prob)
else:
    clf = SVC(C=10.0, gamma="scale", kernel="rbf", probability=True, random_state=SEED, cache_size=2000)
    clf.fit(X_tv_un, y_tv)
    y_pred = clf.predict(X_te_un); y_prob = clf.predict_proba(X_te_un)
m_t1 = compute_metrics(y_te, y_pred, y_prob)
m_t1["model"] = "SVM-RBF + PCA(500) unscaled"
print(f"Tier 1 best: acc={m_t1['accuracy']:.4f}, F1={m_t1['macro_f1']:.4f}, ROC-AUC={m_t1['roc_auc']:.4f} ({time.time()-t0:.1f}s)")


## 4. Tier 2 — Handcrafted Features + Classical

HOG (1764-D, 64×64 win, 16×16 blocks, stride 8, 9 orientations) ⊕ Color histograms (96-D, 32 bins per channel L1) → 1860-D.
Best: SVM-RBF + HOG ≈ 0.831 test acc.


In [ ]:
# HOG + ColorHist extraction
def extract_hog(image):
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    hog = cv2.HOGDescriptor((64,64),(16,16),(8,8),(8,8), 9)
    return hog.compute(gray).flatten()

def extract_hog_batch(imgs):
    return np.array([extract_hog(imgs[i]) for i in range(len(imgs))])

def extract_color_hist(imgs, bins=32):
    hists = []
    for i in range(len(imgs)):
        ch = []
        for c in range(3):
            h, _ = np.histogram(imgs[i,:,:,c], bins=bins, range=(0.0, 1.0))
            h = h.astype(np.float32); h /= h.sum() + 1e-7
            ch.append(h)
        hists.append(np.concatenate(ch))
    return np.array(hists)

print("Extracting HOG + ColorHist (Tier 2 features) ...")
t0 = time.time()
hog_tr = extract_hog_batch(X_tr_img); hog_val = extract_hog_batch(X_val_img); hog_te = extract_hog_batch(X_te_img)
ch_tr = extract_color_hist(X_tr_img); ch_val = extract_color_hist(X_val_img); ch_te = extract_color_hist(X_te_img)
feat_tr = np.concatenate([hog_tr, ch_tr], axis=1)
feat_val = np.concatenate([hog_val, ch_val], axis=1)
feat_te = np.concatenate([hog_te, ch_te], axis=1)
print(f"  HOG+CH features: {feat_tr.shape} ({time.time()-t0:.1f}s)")
sc = StandardScaler()
feat_tv = sc.fit_transform(np.concatenate([feat_tr, feat_val]))
feat_te_s = sc.transform(feat_te)


In [ ]:
# Tier 2: cuML SVM-RBF (or sklearn fallback) on HOG features
print("Training Tier 2 SVM-RBF on HOG+ColorHist ...")
t0 = time.time()
if HAS_CUML:
    clf = CumlSVC(C=10.0, gamma="scale", kernel="rbf", probability=True)
    clf.fit(feat_tv.astype(np.float32), y_tv)
    y_pred = clf.predict(feat_te_s.astype(np.float32))
    y_pred = cp.asnumpy(y_pred) if hasattr(y_pred, "get") else np.asarray(y_pred)
    y_prob = clf.predict_proba(feat_te_s.astype(np.float32))
    y_prob = cp.asnumpy(y_prob) if hasattr(y_prob, "get") else np.asarray(y_prob)
else:
    clf = SVC(C=10.0, gamma="scale", kernel="rbf", probability=True, random_state=SEED, cache_size=2000)
    clf.fit(feat_tv, y_tv)
    y_pred = clf.predict(feat_te_s); y_prob = clf.predict_proba(feat_te_s)
m_t2 = compute_metrics(y_te, y_pred, y_prob)
m_t2["model"] = "SVM-RBF + HOG+ColorHist"
print(f"Tier 2: acc={m_t2['accuracy']:.4f}, F1={m_t2['macro_f1']:.4f}, ROC-AUC={m_t2['roc_auc']:.4f} ({time.time()-t0:.1f}s)")


## 5. Tier 3 — Shallow CNN (391K params)

Toggleable BatchNorm and Dropout2d. Trained on GPU-resident TensorDataset (1 s/epoch on A100).


In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self, num_classes=10, use_batchnorm=False, dropout_rate=0.0):
        super().__init__()
        def block(in_c, out_c, last=False):
            layers = [nn.Conv2d(in_c, out_c, 3, padding=1)]
            if use_batchnorm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.ReLU(inplace=True))
            if not last:
                layers.append(nn.MaxPool2d(2, 2))
                if dropout_rate > 0: layers.append(nn.Dropout2d(dropout_rate))
            return nn.Sequential(*layers)
        self.b1 = block(3, 32); self.b2 = block(32, 64); self.b3 = block(64, 128); self.b4 = block(128, 256, last=True)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop_fc = nn.Dropout(dropout_rate) if dropout_rate > 0 else nn.Identity()
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.b1(x); x = self.b2(x); x = self.b3(x); x = self.b4(x)
        x = self.gap(x).view(x.size(0), -1)
        x = self.drop_fc(x); return self.fc(x)

scnn = ShallowCNN()
print(f"ShallowCNN params: {sum(p.numel() for p in scnn.parameters() if p.requires_grad):,}")


In [ ]:
# Augmentation helper (Kornia if available, else manual)
if HAS_KORNIA:
    aug_geom = torch.nn.Sequential(
        K.RandomHorizontalFlip(p=0.5),
        K.RandomVerticalFlip(p=0.5),
        K.RandomRotation(degrees=15.0),
    ).to(DEVICE)
    def gpu_augment(x): return aug_geom(x)
else:
    def gpu_augment(x):
        B = x.size(0)
        if torch.rand(1, device=x.device) < 0.5: x = torch.flip(x, [3])
        if torch.rand(1, device=x.device) < 0.5: x = torch.flip(x, [2])
        return x
print("Augmentation helper ready.")


In [ ]:
def fast_train_cnn(use_bn=False, dropout=0.0, wd=0.0, augment=False, optimizer_name='Adam',
                    lr=1e-3, num_epochs=NUM_EPOCHS_CNN, seed=SEED, patience=None, verbose=False):
    seed_everything(seed); torch.cuda.manual_seed_all(seed)
    model = ShallowCNN(num_classes=NUM_CLASSES, use_batchnorm=use_bn, dropout_rate=dropout).to(DEVICE)
    opt_factory = {
        'SGD': lambda: torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0, weight_decay=wd),
        'SGD_Momentum': lambda: torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd),
        'NAG': lambda: torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd),
        'Adam': lambda: torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd),
        'Adagrad': lambda: torch.optim.Adagrad(model.parameters(), lr=lr, weight_decay=wd),
        'RMSProp': lambda: torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.99, weight_decay=wd),
    }
    optimizer = opt_factory[optimizer_name]()
    crit = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_vl = float('inf'); best_state = None; best_ep = 0; no_improve = 0
    t0 = time.time()
    for ep in range(1, num_epochs + 1):
        model.train()
        tl, tc, tt = 0.0, 0, 0
        for x, y in gpu_iter(X_tr_t, y_tr_t, BATCH_SIZE, True, seed*1000+ep):
            if augment: x = gpu_augment(x)
            optimizer.zero_grad()
            out = model(x); loss = crit(out, y)
            if not torch.isfinite(loss): return None, history, 0, time.time()-t0, None
            loss.backward(); optimizer.step()
            tl += loss.item()*x.size(0); tc += (out.argmax(1)==y).sum().item(); tt += x.size(0)
        model.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for x, y in gpu_iter(X_val_t, y_val_t, BATCH_SIZE, False, 0):
                out = model(x); loss = crit(out, y)
                vl += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += x.size(0)
        history['train_loss'].append(tl/tt); history['val_loss'].append(vl/vt)
        history['train_acc'].append(tc/tt); history['val_acc'].append(vc/vt)
        if verbose and ep % 5 == 0:
            print(f"  ep {ep:3d}: tr {tl/tt:.3f}/{tc/tt:.4f} | va {vl/vt:.3f}/{vc/vt:.4f}")
        if vl/vt < best_vl:
            best_vl = vl/vt; best_state = copy.deepcopy(model.state_dict()); best_ep = ep; no_improve = 0
        else:
            no_improve += 1
            if patience is not None and no_improve >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    all_y, all_p, all_pr = [], [], []
    with torch.no_grad():
        for x, y in gpu_iter(X_te_t, y_te_t, BATCH_SIZE, False, 0):
            out = model(x); probs = torch.softmax(out, 1)
            all_y.append(y.cpu().numpy()); all_p.append(out.argmax(1).cpu().numpy()); all_pr.append(probs.cpu().numpy())
    return model, history, best_ep, time.time()-t0, compute_metrics(np.concatenate(all_y), np.concatenate(all_p), np.concatenate(all_pr))

# ShallowCNN baseline run (Adam lr=1e-3, no reg)
print(f"ShallowCNN baseline: Adam lr=1e-3, {NUM_EPOCHS_CNN} epochs ...")
res = fast_train_cnn(optimizer_name='Adam', lr=1e-3, num_epochs=NUM_EPOCHS_CNN, seed=SEED, verbose=True)
m_t3_baseline = res[4]; m_t3_baseline['model'] = 'ShallowCNN baseline'
print(f"Tier 3 baseline: acc={m_t3_baseline['accuracy']:.4f} ({res[3]:.1f}s, {res[2]} best ep)")


## 6. Experiment B — Optimizer Ablation

`RUN_MODE="full"`: 6 optimizers × 3 LRs × 3 seeds = 54 runs (~88 min on A100).
`RUN_MODE="quick"`: 6 optimizers × 1 LR (default best per optimizer) × 1 seed = 6 runs (~10 min on A100).


In [ ]:
OPTIMIZERS = ['SGD', 'SGD_Momentum', 'NAG', 'Adam', 'Adagrad', 'RMSProp']
LR_GRID = [1e-4, 1e-3, 1e-2] if RUN_MODE == "full" else [1e-3]
DEFAULT_LR_PER_OPT = {'SGD': 1e-2, 'SGD_Momentum': 1e-2, 'NAG': 1e-2,
                      'Adam': 1e-3, 'Adagrad': 1e-2, 'RMSProp': 1e-3}

print(f"Exp B: {len(OPTIMIZERS)} optimizers × {len(LR_GRID)} LRs × {len(SEEDS)} seeds = {len(OPTIMIZERS)*len(LR_GRID)*len(SEEDS)} runs")
expB_results = []
for opt in OPTIMIZERS:
    lrs = LR_GRID if RUN_MODE == "full" else [DEFAULT_LR_PER_OPT[opt]]
    for lr in lrs:
        for seed in SEEDS:
            print(f"  [{opt} lr={lr} seed={seed}] ...", end=" ", flush=True)
            res = fast_train_cnn(optimizer_name=opt, lr=lr, num_epochs=NUM_EPOCHS_CNN, seed=seed)
            if res[0] is None:
                expB_results.append({"optimizer": opt, "lr": lr, "seed": seed, "test_acc": None, "diverged": True})
                print("DIVERGED")
            else:
                _, hist, best_ep, t, m = res
                conv90 = next((i+1 for i,a in enumerate(hist['val_acc']) if a >= 0.90), None)
                expB_results.append({
                    "optimizer": opt, "lr": lr, "seed": seed,
                    "test_acc": m["accuracy"], "macro_f1": m["macro_f1"],
                    "best_epoch": best_ep, "convergence_epoch_90": conv90,
                    "train_time_sec": t, "diverged": False,
                })
                print(f"acc={m['accuracy']:.4f} ({t:.1f}s)")

dfB = pd.DataFrame(expB_results)
dfB.to_csv(os.path.join(TABLES_DIR, 'exp_b_results.csv'), index=False)
print(f"\nExp B summary:")
print(dfB.groupby('optimizer')['test_acc'].agg(['mean','std','count']).sort_values('mean', ascending=False).to_string())
BEST_OPT_ROW = dfB.groupby(['optimizer','lr'])['test_acc'].mean().idxmax()
BEST_OPTIMIZER, BEST_LR = BEST_OPT_ROW
print(f"\n>>> Best optimizer: {BEST_OPTIMIZER} lr={BEST_LR}")


## 7. Experiment C — Regularization Ablation

`RUN_MODE="full"`: 9 configs × 3 seeds = 27 runs (~66 min on A100).
`RUN_MODE="quick"`: 9 configs × 1 seed = 9 runs (~15 min).


In [ ]:
CONFIGS = {
    "1_Baseline":     {"bn": False, "dropout": 0.0, "wd": 0.0,  "augment": False, "es": False},
    "2a_L2_1e-4":     {"bn": False, "dropout": 0.0, "wd": 1e-4, "augment": False, "es": False},
    "3a_Dropout_0.3": {"bn": False, "dropout": 0.3, "wd": 0.0,  "augment": False, "es": False},
    "4_BatchNorm":    {"bn": True,  "dropout": 0.0, "wd": 0.0,  "augment": False, "es": False},
    "5_DataAug":      {"bn": False, "dropout": 0.0, "wd": 0.0,  "augment": True,  "es": False},
    "6_EarlyStop":    {"bn": False, "dropout": 0.0, "wd": 0.0,  "augment": False, "es": True},
    "7_AllCombined":  {"bn": True,  "dropout": 0.3, "wd": 1e-4, "augment": True,  "es": True},
}

print(f"Exp C: {len(CONFIGS)} configs × {len(SEEDS)} seeds = {len(CONFIGS)*len(SEEDS)} runs")
expC_results = []
for name, cfg in CONFIGS.items():
    for seed in SEEDS:
        print(f"  [{name} seed={seed}] ...", end=" ", flush=True)
        patience = EARLY_STOP_PATIENCE if cfg['es'] else None
        res = fast_train_cnn(use_bn=cfg['bn'], dropout=cfg['dropout'], wd=cfg['wd'],
                             augment=cfg['augment'], optimizer_name=BEST_OPTIMIZER, lr=BEST_LR,
                             num_epochs=NUM_EPOCHS_CNN, seed=seed, patience=patience)
        if res[0] is None:
            expC_results.append({"config": name, "seed": seed, "test_acc": None, "diverged": True})
            print("DIVERGED")
        else:
            _, hist, best_ep, t, m = res
            gap = hist['train_acc'][-1] - m['accuracy']
            expC_results.append({
                "config": name, "seed": seed,
                "test_acc": m["accuracy"], "macro_f1": m["macro_f1"],
                "final_train_acc": hist['train_acc'][-1],
                "overfitting_gap": gap, "best_epoch": best_ep,
                "epochs_run": len(hist['train_acc']), "train_time_sec": t,
            })
            print(f"acc={m['accuracy']:.4f} gap={gap:+.4f} ({t:.1f}s)")

dfC = pd.DataFrame(expC_results)
dfC.to_csv(os.path.join(TABLES_DIR, 'exp_c_results.csv'), index=False)
print(f"\nExp C summary:")
print(dfC.groupby('config')['test_acc'].agg(['mean','std','count']).to_string())


## 8. Tier 4 — ResNet-18 Fine-tuning

ImageNet-pretrained ResNet-18, full fine-tune. Inputs upsampled 64→224, ImageNet-normalized.
`RUN_MODE="quick"`: 1 seed × 15 ep ≈ 5 min. `RUN_MODE="full"`: 3 seeds × 50 ep ≈ 15 min.


In [ ]:
# ResNet-18 transform helpers
EUROSAT_M_T = torch.tensor(EUROSAT_MEAN, device=DEVICE).view(1,3,1,1)
EUROSAT_S_T = torch.tensor(EUROSAT_STD, device=DEVICE).view(1,3,1,1)
IMAGENET_M_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1,3,1,1)
IMAGENET_S_T = torch.tensor(IMAGENET_STD, device=DEVICE).view(1,3,1,1)

def to_resnet_input(x_eurosat_norm, augment=False):
    x = x_eurosat_norm * EUROSAT_S_T + EUROSAT_M_T
    if augment and HAS_KORNIA:
        x = aug_geom(x)  # geometric only; ColorJitter would need separate stack
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    x = (x - IMAGENET_M_T) / IMAGENET_S_T
    return x

def get_resnet18():
    weights = tvmodels.ResNet18_Weights.IMAGENET1K_V1
    model = tvmodels.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

print("ResNet-18 helpers ready.")


In [ ]:
def fast_train_resnet(seed=SEED, num_epochs=NUM_EPOCHS_RESNET, patience=EARLY_STOP_PATIENCE,
                       use_amp=True, augment=True):
    seed_everything(seed); torch.cuda.manual_seed_all(seed)
    model = get_resnet18().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_vl = float('inf'); best_state = None; best_ep = 0; no_improve = 0
    t0 = time.time()
    for ep in range(1, num_epochs + 1):
        model.train()
        tl, tc, tt = 0.0, 0, 0
        for x, y in gpu_iter(X_tr_t, y_tr_t, BATCH_SIZE, True, seed*1000+ep):
            x_in = to_resnet_input(x, augment=augment)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                out = model(x_in); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tl += loss.item()*x.size(0); tc += (out.argmax(1)==y).sum().item(); tt += x.size(0)
        model.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for x, y in gpu_iter(X_val_t, y_val_t, BATCH_SIZE, False, 0):
                x_in = to_resnet_input(x, augment=False)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    out = model(x_in); loss = crit(out, y)
                vl += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += x.size(0)
        history['train_loss'].append(tl/tt); history['val_loss'].append(vl/vt)
        history['train_acc'].append(tc/tt); history['val_acc'].append(vc/vt)
        print(f"  ep {ep:3d}: tr {tl/tt:.3f}/{tc/tt:.4f} | va {vl/vt:.3f}/{vc/vt:.4f}")
        if vl/vt < best_vl:
            best_vl = vl/vt; best_state = copy.deepcopy(model.state_dict()); best_ep = ep; no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience: print(f"  early stop @ ep {ep} (best ep {best_ep})"); break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    all_y, all_p, all_pr = [], [], []
    with torch.no_grad():
        for x, y in gpu_iter(X_te_t, y_te_t, BATCH_SIZE, False, 0):
            x_in = to_resnet_input(x, augment=False).float()
            out = model(x_in).float(); probs = torch.softmax(out, 1)
            all_y.append(y.cpu().numpy()); all_p.append(out.argmax(1).cpu().numpy()); all_pr.append(probs.cpu().numpy())
    return model, history, best_ep, time.time()-t0, compute_metrics(np.concatenate(all_y), np.concatenate(all_p), np.concatenate(all_pr))

resnet_results = []
for seed in SEEDS:
    print(f"\nResNet-18 seed={seed} ...")
    res = fast_train_resnet(seed=seed, num_epochs=NUM_EPOCHS_RESNET, patience=EARLY_STOP_PATIENCE)
    if res[0] is None:
        print(f"  DIVERGED")
        continue
    _, hist, best_ep, t, m = res
    resnet_results.append({"seed": seed, **{k: m[k] for k in ['accuracy','macro_f1','roc_auc']},
                           "best_epoch": best_ep, "train_time_sec": t})

dfR = pd.DataFrame(resnet_results)
print(f"\n=== Tier 4 ResNet-18 ({len(resnet_results)} seeds) ===")
print(dfR.to_string(index=False))
m_t4 = {"model": "ResNet-18 fine-tuned",
        "accuracy": dfR['accuracy'].mean(), "accuracy_std": dfR['accuracy'].std() if len(dfR)>1 else 0.0,
        "macro_f1": dfR['macro_f1'].mean(), "roc_auc": dfR['roc_auc'].mean()}
print(f"\nMean acc: {m_t4['accuracy']:.4f} ± {m_t4['accuracy_std']:.4f}")


## 9. Final Comparison and Per-Class Analysis

Build the master comparison table. Apply paired t-tests on 3-seed triples (only meaningful in `RUN_MODE="full"`).


In [ ]:
from scipy import stats

def best_per_optimizer_acc():
    return dfB.groupby('optimizer')['test_acc'].mean().sort_values(ascending=False)

def best_exp_c_acc():
    return dfC.groupby('config')['test_acc'].mean().sort_values(ascending=False)

# Build comparison table
rows = [
    {"Model": "LR + PCA(200)", "Tier": 1, "Acc": m_lr['accuracy'], "F1": m_lr['macro_f1']},
    {"Model": "SVM-RBF + PCA(500)", "Tier": 1, "Acc": m_t1['accuracy'], "F1": m_t1['macro_f1']},
    {"Model": "SVM-RBF + HOG", "Tier": 2, "Acc": m_t2['accuracy'], "F1": m_t2['macro_f1']},
    {"Model": "ShallowCNN baseline", "Tier": 3, "Acc": m_t3_baseline['accuracy'], "F1": m_t3_baseline['macro_f1']},
]
if len(dfC) > 0:
    bn_row = dfC[dfC['config'] == '4_BatchNorm']
    if len(bn_row) > 0:
        rows.append({"Model": "ShallowCNN + BN", "Tier": 3,
                     "Acc": bn_row['test_acc'].mean(), "F1": bn_row['macro_f1'].mean()})
rows.append({"Model": "ResNet-18 (fine-tuned)", "Tier": 4,
             "Acc": m_t4['accuracy'], "F1": m_t4['macro_f1']})
df_final = pd.DataFrame(rows)
df_final.to_csv(os.path.join(TABLES_DIR, 'exp_a_final_comparison.csv'), index=False)
print("=== Final tier comparison ===")
print(df_final.to_string(index=False))

tier_max = df_final.groupby('Tier')['Acc'].max()
print(f"\nTier max: {dict(tier_max.round(4))}")
print(f"Monotonic: {tier_max.is_monotonic_increasing}")

# Save Tier 4 confusion matrix and final comparison plot
import seaborn as sns
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x='Model', y='Acc', hue='Tier', data=df_final, ax=ax, palette='viridis')
ax.tick_params(axis='x', rotation=20)
ax.set_ylim([0.4, 1.0])
for i, v in enumerate(df_final['Acc']):
    ax.text(i, v+0.005, f"{v:.3f}", ha='center', fontsize=9)
ax.set_title("Tier comparison (test accuracy)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'exp_a_tier_comparison.png'), dpi=120)
plt.show()


In [ ]:
# Save run results to JSON for downstream analysis
def to_json_safe(obj):
    if isinstance(obj, (np.integer, np.int64)): return int(obj)
    if isinstance(obj, (np.floating, np.float64)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list): return [to_json_safe(v) for v in obj]
    return obj

summary = {
    "run_mode": RUN_MODE,
    "seeds_used": SEEDS,
    "tier1_best": to_json_safe(m_t1),
    "tier2_best": to_json_safe(m_t2),
    "tier3_baseline": to_json_safe(m_t3_baseline),
    "tier4_resnet": to_json_safe(m_t4),
    "exp_b_runs": len(dfB),
    "exp_c_runs": len(dfC),
    "monotonic": bool(tier_max.is_monotonic_increasing),
    "eurosat_mean": EUROSAT_MEAN,
    "eurosat_std": EUROSAT_STD,
}
with open(os.path.join(RESULTS_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


## 10. Conclusions

If `RUN_MODE="full"`, the numbers above should match those in the report:

- **Tier 1 best:** SVM-RBF + PCA(500) unscaled = 0.6872
- **Tier 2 best:** SVM-RBF + HOG = 0.8309
- **Tier 3 baseline:** ShallowCNN (Adam lr=1e-3) = 0.9550 ± 0.0034
- **Tier 3 best:** ShallowCNN + BatchNorm = 0.9657 ± 0.0013
- **Tier 4:** ResNet-18 fine-tuned = 0.9816 ± 0.0016

In `RUN_MODE="quick"` (1 seed each, fewer epochs), expect numbers within ±1pp of full-mode means.

Key findings (confirmed in full mode):
1. **Representation learning is the win**: T2→T3 jump is +12.4pp; T3→T4 only +1.6pp despite 28× params.
2. **Optimizer choice is small at the top**: top 4 optimizers tie within 0.3pp; Adam wins on convergence speed (10 ep to 90% val acc vs 13–21 for others).
3. **BatchNorm alone beats stacking**: AllCombined regularization underperforms BN-alone by 3pp due to over-regularization.

See [`report/final_report.pdf`](../report/final_report.pdf) for full analysis.
